# DFU Reliable V2 — Rescue and Resume
Preserves completed V2 trials, removes failed temporary checkpoint files, frees the duplicated active backup, and resumes the interrupted trial with a storage-bounded checkpoint writer.

In [ ]:
import os, sys, shutil, subprocess, json, time, uuid
from pathlib import Path
from google.colab import drive

MOUNT=Path('/content/drive'); MY=MOUNT/'MyDrive'
if not MY.is_dir(): drive.mount(str(MOUNT), force_remount=False)
if not MY.is_dir(): raise RuntimeError('Google Drive unavailable; rescue not started.')
probe=MY/'DFU-ImageGuard'/'_mount_verification'/f'rescue_{uuid.uuid4().hex}.txt'
probe.parent.mkdir(parents=True,exist_ok=True); value=str(time.time_ns()); probe.write_text(value)
if probe.read_text()!=value: raise RuntimeError('Drive write/read verification failed.')
probe.unlink(); print('Drive write/read verification: PASS')

subprocess.run([sys.executable,'-m','pip','install','-q','timm>=1.0.9','kagglehub>=0.3','ImageHash>=4.3','scikit-learn>=1.5','scipy>=1.13','matplotlib>=3.9','pandas>=2.2','Pillow>=10.4','tabulate>=0.9'],check=True)
REPO='https://github.com/AzizulHakim00/DFU-ImageGuard.git'
CODE_COMMIT='613f7150cb8957222a92ed2b41058780d7496435'
WORK=Path('/content/DFU-ImageGuard-v2-rescue')
if WORK.exists(): shutil.rmtree(WORK)
subprocess.run(['git','clone','--filter=blob:none','--no-checkout',REPO,str(WORK)],check=True)
subprocess.run(['git','-C',str(WORK),'checkout',CODE_COMMIT],check=True)
os.chdir(WORK); sys.path.insert(0,str(WORK))
for module_name in list(sys.modules):
    if module_name=='src' or module_name.startswith('src.'):
        del sys.modules[module_name]
loaded=subprocess.run(['git','rev-parse','HEAD'],capture_output=True,text=True,check=True).stdout.strip()
if loaded!=CODE_COMMIT: raise RuntimeError(f'Commit mismatch: {loaded} != {CODE_COMMIT}')
print('Loaded V2 storage-rescue commit:',loaded)

from src import reliable_runner_v2 as runner
from src.reliable_storage_rescue import (
    ORIGINAL_TRAINING_COMMIT,
    install_rescue_patches,
    prepare_existing_v2_run,
)

audit=prepare_existing_v2_run(
    drive_root='/content/drive/MyDrive/DFU-ImageGuard',
    backup_root='/content/drive/MyDrive/DFU-ImageGuard-Backup',
    run_id='RELIABLE_DFU_CV_V2',
)
install_rescue_patches(runner)
print('Storage rescue patches: INSTALLED')
print('Resume action:',audit['resume_action'])
settings=runner.ReliableSettingsV2(
    run_id='RELIABLE_DFU_CV_V2',
    seeds=(2026,2027,2028),
    folds=(0,1,2,3,4),
    models=('convnextv2_tiny','mobilenetv3_large','densenet121'),
    max_epochs=30,
    patience=7,
    batch_size=16,
    num_workers=2,
    target_sensitivity=.95,
    source_commit=ORIGINAL_TRAINING_COMMIT,
)
result=runner.run_reliable_framework_v2(settings)
print(json.dumps(result,indent=2,default=str))
